# Brakes

Hardware to torque and balance, and rotor temperature (`lapsim/subsystems/brakes.py`). The balance table sets
the front share the hydraulics give against the share that would use both axles fully (load transfer and
aero included), so it says which axle locks first at each deceleration. Rotor temperature is one lumped mass
per rotor with speed-dependent convection and radiation; repeated identical stops show whether it levels off.

In [ ]:
%matplotlib inline
from nb import *

In [ ]:
car_pick = W.ToggleButtons(options=list(CARS), description="Car")
stops = [W.FloatSlider(value=100, min=60, max=130, step=5, description="from [km/h]", continuous_update=False),
         W.FloatSlider(value=30, min=0, max=60, step=5, description="to [km/h]", continuous_update=False),
         W.IntSlider(value=15, min=3, max=40, description="stops", continuous_update=False),
         W.FloatSlider(value=1.4, min=0.6, max=2.0, step=0.05, description="decel [g]", continuous_update=False)]
out = W.Output()
box = W.VBox()


def build(_=None):
    global CAR, SL
    CAR = lapsim.load_car(CARS[car_pick.value])
    SL = [slider(CAR, "vehicle:brakes.front_bias", 0.45, 0.80, 0.01, "front bias")]
    for s in SL + stops:
        s.observe(draw, "value")
    box.children = [*SL, *stops]
    draw()


def draw(*_):
    with out:
        out.clear_output(wait=True)
        veh = with_values(CAR, SL).build()
        B = lapsim.subsystems.brakes
        display(pd.DataFrame(B.balance_table(veh, decel_g=(0.4, 0.8, 1.2, 1.6, 2.0))).round(3))
        df, peaks = B.repeated_stops(veh, stops[0].value / 3.6, stops[1].value / 3.6, stops[2].value, stops[3].value)
        fig, ax = plt.subplots(figsize=(12, 3.4))
        ax.plot(df["t"], df["T_front_C"], label="front rotor"); ax.plot(df["t"], df["T_rear_C"], label="rear rotor")
        ax.set(xlabel="time [s]", ylabel="degC", title="Rotor temperature over repeated stops"); ax.legend()
        plt.show()


car_pick.observe(build, "value")
build()
display(W.VBox([car_pick, box, out]))